# Small parameter scan

Independent quick scan with matched seeded configurations, realization-level uncertainty, and a heatmap. Increase sizes and sample counts only after checking runtime. This notebook does not read another notebook’s state.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
from src.lattice import square_lattice_hamiltonian, square_lattice_dispersion, momentum_grid
from src.gap_equation import uniform_gap
from src.solver import solve_bdg
from src.disorder import onsite_disorder
from src.hamiltonian import gaussian_dos
print("Project:", ROOT)


In [ ]:
L, t, mu, U, T = 6, 1., -0.5, 2.2, 0.08
Ws, tps, nreal, seed0 = [0.,.8], [0.,.2], 2, 7341
rows=[]
for tp in tps:
    for W in Ws:
        for r in range(nreal):
            seed=seed0+int(W*100)+r
            V=onsite_disorder(L*L,W,"uniform",seed)
            result=solve_bdg(square_lattice_hamiltonian(L,t,tp,mu,V),U,T,initial_gap=.3,tolerance=5e-6,max_iterations=300,mixing=.55)
            a=np.abs(result.gap)
            rows.append(dict(t_prime=tp,W=W,realization=r,seed=seed,mean_gap=a.mean(),spatial_std=a.std(),converged=result.converged,iterations=result.iterations))
import pandas as pd
raw=pd.DataFrame(rows); display(raw)
summary=raw.groupby(["t_prime","W"]).agg(mean_gap=("mean_gap","mean"),sd=("mean_gap","std"),mean_spatial_std=("spatial_std","mean"),samples=("realization","count")).reset_index()
summary["sem"]=summary.sd/np.sqrt(summary.samples); display(summary)


In [ ]:
for tp in tps:
    q=summary[summary.t_prime==tp]
plt.errorbar(q["W"],q["mean_gap"],yerr=q["sem"],marker="o",capsize=3,label=f"t'/t={tp:g}")
plt.xlabel("Uniform disorder width W/t"); plt.ylabel("Mean |Δi|/t"); plt.legend(); plt.grid(alpha=.3); plt.show()
heat=summary.pivot(index="t_prime",columns="W",values="mean_gap")
plt.imshow(heat.values,aspect="auto",origin="lower"); plt.xticks(range(len(heat.columns)),heat.columns); plt.yticks(range(len(heat.index)),heat.index); plt.xlabel("W/t grid index"); plt.ylabel("t'/t grid index"); plt.colorbar(label="Mean |Δi|/t"); plt.show()
